# Qualitative comparison of saved predictions

Compare completed evaluations on the same samples without loading models or running inference.
Run the cells in order using the existing GeoTest environment. Paths below point to the
completed seed-12 evaluations. Ground truth and predictions are read only from saved NPZ files.

The figures describe selected examples, not aggregate performance. Random selection uses
a dedicated local seed; manual examples should be identified as such in presentation captions.


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import yaml
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# Works when the notebook is opened from the project root or experiments/.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src/plotting.py").is_file() and (p / "run.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the GeoTest repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.plotting import scientific_style, compact_legend_layout, save_figure

import pandas as pd
import torch
from IPython.display import display
from src.evaluation.metrics import SegmentationMetrics


## Evaluation directories
Use evaluation directories, not training-run directories. Change `SPLIT` to `bolivia` to compare those exports. No newest-run discovery is performed.


In [ ]:
EVALUATION_DIRS = [
    "runs/sen1floods11/croma/linear/croma_s1s2_linear_100_noshift/lr0.003_seed12/20261124_024245_a9568fd2/evaluations/20261127_145628_4d9395b4",
    "runs/sen1floods11/terramind/linear/terramind_s1s2_linear_100_noshift/lr0.005_seed12/20261124_012716_c51c2c53/evaluations/20261127_145931_7e5765f4",
    "runs/sen1floods11/baselines/unet/unet_s1s2_100_noshift/lr0.0003_seed12/20261123_235140_192c3f86/evaluations/20261127_150101_ec959812",
    "runs/sen1floods11/baselines/resnet18_unet/resnet18_unet_s1s2_100_noshift/lr0.001_seed12/20261124_004303_3f180faa/evaluations/20261127_150132_46d8aea9",
]
SPLIT = "test"


## Column labels
Labels follow the same order as the evaluation directories; any number of runs is supported.


In [ ]:
RUN_LABELS = ["CROMA", "TerraMind", "U-Net", "ResNet-18 U-Net"]


## Sample selection
Set `SAMPLE_IDS` to exact IDs, or leave it empty to select `N_SAMPLES` from the sorted common pool. The local seed does not alter any training/evaluation random stream.


In [ ]:
SAMPLE_IDS = []  # Example: ["Ghana_313799"] only if present in the selected split.
N_SAMPLES = 5

SELECTION_SEED = 12


## Output and optional RGB
RGB is visual context, not necessarily model input. Existing RGB_PATHS and RGB_BANDS
remain explicit overrides. AUTO_RGB looks up Sen1Floods11 optical and label rasters
using the dataset's HAND_DIR and actual sample IDs.

The inspected Ghana_313799_S2Hand.tif stores B1, B2, B3, B4, B5, B6, B7, B8,
B8A, B9, B10, B11, B12. The loader resolves B4/B3/B2 from each raster's band
descriptions (one-based RGB indices 4/3/2 in this file), verifies CRS, transform,
dimensions and source-label equality against the saved target. It never resamples.
For manually supplied RGB paths, RGB_BANDS is used and spatial alignment remains
the user's responsibility; dimensions are checked.

RGB alone receives a per-band 2-98 percentile display stretch. If any selected
image is unavailable or fails checks, the whole RGB column is omitted with a warning.
Set AUTO_RGB=False and leave RGB_PATHS empty to use masks only.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "reports/qualitative_predictions/seed12" / SPLIT
SAVE_FIGURES = True
SHOW_LEGEND = True
MAKE_ERROR_FIGURE = False
LEGEND_COLUMNS = 4
PANEL_INCHES = 1.65

RGB_PATHS = {}  # {"sample_id": "data/path/to/aligned_optical.tif"}
RGB_BANDS = (4, 3, 2)

# Displayed metrics are per chip, never pooled split-level values.
DISPLAY_METRICS = ("miou", "water_iou")  # Use () to hide annotations.
WATER_CLASS_ID = "auto"  # Resolve a class named "water"; None disables it.
CROSS_CHECK_REPORTS = True
AUTO_RGB = True  # Discover Sen1Floods11 paths; RGB_PATHS overrides individual chips.


## Loading and compatibility checks
Archives use `allow_pickle=False`. IDs are read from archive contents, not filenames or loader order. All common samples are checked one at a time, including exact ground-truth equality; only selected arrays are retained for plotting. Unnamed classes are shown as numeric IDs rather than guessed meanings.


In [ ]:
def project_path(value):
    path = Path(value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def load_run(directory, label):
    directory = project_path(directory)
    config = yaml.safe_load((directory / "config.yaml").read_text(encoding="utf-8"))
    mask_dir = directory / "prediction_masks" / SPLIT
    metadata = json.loads((mask_dir / "metadata.json").read_text(encoding="utf-8"))
    if metadata.get("completed") is not True:
        raise ValueError(f"{label}: mask export is not complete.")
    dataset = config.get("dataset", {}).get("name")
    if not dataset or metadata.get("split") != SPLIT:
        raise ValueError(f"{label}: missing dataset name or inconsistent split.")
    nclasses = metadata["num_classes"]
    mapping = metadata["class_mapping"]
    if nclasses < 2 or set(mapping) != {str(i) for i in range(nclasses)}:
        raise ValueError(f"{label}: invalid class mapping.")
    index = {}
    for path in sorted(mask_dir.glob("*.npz")):
        with np.load(path, allow_pickle=False) as archive:
            if not {"sample_id", "prediction", "target"} <= set(archive.files):
                raise ValueError(f"Missing arrays: {path}")
            stored_id = archive["sample_id"]
            if stored_id.shape != () or stored_id.dtype.kind not in "US":
                raise ValueError(f"Expected a scalar string sample ID: {path}")
            sample_id = str(stored_id.item())
        if not sample_id or sample_id in index:
            raise ValueError(f"{label}: empty or duplicated ID {sample_id!r}")
        index[sample_id] = path
    if not index or len(index) != metadata.get("num_samples"):
        raise ValueError(f"{label}: mask count differs from completed export metadata.")
    return dict(directory=directory, label=label, dataset=dataset,
                metadata=metadata, index=index)


def load_masks(run, sample_id):
    with np.load(run["index"][sample_id], allow_pickle=False) as archive:
        target = archive["target"]
        prediction = archive["prediction"]
        if str(archive["sample_id"].item()) != sample_id:
            raise ValueError(f"Sample ID changed: {sample_id}")
    if (target.ndim != 2 or prediction.shape != target.shape or
            target.dtype.kind not in "iu" or prediction.dtype.kind not in "iu"):
        raise ValueError(f"{run['label']}/{sample_id}: incompatible mask shapes or dtypes.")
    nclasses = run["metadata"]["num_classes"]
    valid = target != run["metadata"]["ignore_index"]
    if (np.any((prediction < 0) | (prediction >= nclasses)) or
            np.any((target[valid] < 0) | (target[valid] >= nclasses))):
        raise ValueError(f"{run['label']}/{sample_id}: invalid class identifier.")
    return target, prediction


def check_runs(runs):
    reference = runs[0]
    for run in runs[1:]:
        if run["dataset"] != reference["dataset"]:
            raise ValueError("Runs use different downstream datasets.")
        for key in ("split", "num_classes", "class_mapping", "ignore_index"):
            if run["metadata"][key] != reference["metadata"][key]:
                raise ValueError(f"{run['label']}: incompatible {key}.")
    common = sorted(set.intersection(*(set(run["index"]) for run in runs)))
    if not common:
        raise ValueError("No sample IDs are common to all runs.")
    for sample_id in common:
        target, _ = load_masks(reference, sample_id)
        for run in runs[1:]:
            other, _ = load_masks(run, sample_id)
            if target.shape != other.shape or not np.array_equal(target, other):
                raise ValueError(f"Ground truth differs for {sample_id}: {run['label']}")
    return common


def rgb_location(sample_id):
    if sample_id in RGB_PATHS:
        return project_path(RGB_PATHS[sample_id]), None
    if not AUTO_RGB or runs[0]["dataset"].lower() != "sen1floods11":
        raise ValueError("No RGB source configured.")
    from src.data.datasets.sen1floods11 import HAND_DIR
    return (project_path(HAND_DIR / "S2Hand" / f"{sample_id}_S2Hand.tif"),
            project_path(HAND_DIR / "LabelHand" / f"{sample_id}_LabelHand.tif"))


def load_rgb(sample_id, target):
    import rasterio
    path, label_path = rgb_location(sample_id)
    with rasterio.open(path) as raster:
        if raster.shape != target.shape:
            raise ValueError("RGB and saved mask dimensions differ.")
        bands = tuple(RGB_BANDS)
        if label_path is not None:
            # Read band descriptions from each source, rather than assume ordering.
            descriptions = {str(name).upper(): i for i, name in enumerate(raster.descriptions, 1)
                            if name is not None}
            bands = tuple(next((descriptions[name] for name in aliases if name in descriptions), 0)
                          for aliases in (("B4", "B04"), ("B3", "B03"), ("B2", "B02")))
            if 0 in bands:
                raise ValueError("Cannot identify B4/B3/B2 in the optical band descriptions.")
            with rasterio.open(label_path) as labels:
                if (raster.crs is None or raster.crs != labels.crs or
                        raster.shape != labels.shape or
                        not raster.transform.almost_equals(labels.transform, precision=1e-10)):
                    raise ValueError("Optical and label rasters are not on the same grid.")
                original_target = labels.read(1).astype(np.int64)
                original_target[original_target == -1] = 255  # Existing dataset convention.
                if not np.array_equal(original_target, target):
                    raise ValueError("Source label raster differs from the saved evaluation target.")
        rgb = raster.read(list(bands), masked=True).astype(np.float32).filled(np.nan)
    if rgb.shape != (3, *target.shape):
        raise ValueError("RGB requires three aligned bands.")
    stretched = np.zeros_like(rgb)
    for i, band in enumerate(rgb):
        finite = np.isfinite(band)
        if not finite.any():
            raise ValueError("RGB band contains no finite pixels.")
        low, high = np.percentile(band[finite], [2, 98])
        if high > low:
            stretched[i, finite] = np.clip((band[finite] - low) / (high - low), 0, 1)
    rgb_sources[sample_id] = {"path": str(path), "bands": list(bands),
                              "alignment_checked": label_path is not None}
    return stretched.transpose(1, 2, 0)


## Load runs and select common chips
This cell reads saved masks only. Different split hashes are reported; comparison remains restricted to the exact common IDs with identical ground truth.


In [ ]:
if not EVALUATION_DIRS or len(EVALUATION_DIRS) != len(RUN_LABELS):
    raise ValueError("Provide one label for every evaluation directory.")
if len(set(RUN_LABELS)) != len(RUN_LABELS):
    raise ValueError("Use distinct column labels.")
runs = [load_run(path, label) for path, label in zip(EVALUATION_DIRS, RUN_LABELS)]
common_ids = check_runs(runs)
for run in runs:
    meta = run["metadata"]
    print(f"{run['label']}: {len(run['index'])} masks; seed {meta.get('seed')}; "
          f"checkpoint epoch {meta.get('selected_epoch')}")
if len({run["metadata"].get("split_sha256") for run in runs}) > 1:
    warnings.warn("Split files differ; comparisons use only verified common samples.")
print(f"Common samples: {len(common_ids)}")
print("Available IDs:", common_ids)

if SAMPLE_IDS:
    if len(SAMPLE_IDS) != len(set(SAMPLE_IDS)):
        raise ValueError("Manual sample IDs must be unique.")
    missing = set(SAMPLE_IDS) - set(common_ids)
    if missing:
        raise ValueError(f"Requested IDs not common to all runs: {sorted(missing)}")
    selected_ids = list(SAMPLE_IDS)
else:
    if not 1 <= N_SAMPLES <= len(common_ids):
        raise ValueError("N_SAMPLES must be between 1 and the common sample count.")
    selected_ids = np.random.default_rng(SELECTION_SEED).choice(
        common_ids, size=N_SAMPLES, replace=False).tolist()
print("Selected:", selected_ids)

samples = {sid: [load_masks(run, sid) for run in runs] for sid in selected_ids}
rgb_images = {}
rgb_sources = {}
if RGB_PATHS or AUTO_RGB:
    try:
        rgb_images = {sid: load_rgb(sid, samples[sid][0][0]) for sid in selected_ids}
    except (ImportError, OSError, KeyError, ValueError, IndexError) as exc:
        warnings.warn(f"RGB column omitted: {exc}")
        rgb_images = {}
        rgb_sources = {}


## Per-chip metrics and summary table
Uses GeoTest's SegmentationMetrics on CPU, without loading models. All class IoUs
are retained as ratios in metrics_table and per_chip_metrics.csv. Undefined values
remain missing and display as N/A; they are never replaced with zero or one.
Optional report cross-checks use sample IDs and compare unrounded values.


In [ ]:
# Ratios are retained in memory and CSV; only displayed annotations use percentages.
metrics_by_sample = {}
metric_rows = []
meta = runs[0]["metadata"]
mapping = meta["class_mapping"]
water_id = WATER_CLASS_ID
if water_id == "auto":
    water_ids = [int(key) for key, name in mapping.items()
                 if name is not None and name.strip().lower() == "water"]
    water_id = water_ids[0] if len(water_ids) == 1 else None
if water_id is not None and (type(water_id) is not int or str(water_id) not in mapping):
    raise ValueError("WATER_CLASS_ID must be 'auto', None, or a valid integer class ID.")
if set(DISPLAY_METRICS) - {"miou", "water_iou"}:
    raise ValueError("DISPLAY_METRICS supports miou and water_iou.")

for model_index, run in enumerate(runs):
    report_rows = {}
    report_path = run["directory"] / "metrics" / f"{SPLIT}.json"
    if CROSS_CHECK_REPORTS and report_path.is_file():
        report = json.loads(report_path.read_text(encoding="utf-8"))
        for entry in report.get("per_sample", []):
            sid = str(entry["sample_id"])
            if sid in report_rows:
                raise ValueError(f"Duplicate report ID: {sid}")
            report_rows[sid] = entry
    for sid in selected_ids:
        target, prediction = samples[sid][model_index]
        metric = SegmentationMetrics(num_classes=meta["num_classes"],
                                     ignore_index=meta["ignore_index"])
        metric.update(torch.from_numpy(prediction), torch.from_numpy(target))
        result = metric.compute()
        metrics_by_sample[(sid, model_index)] = result
        row = {"sample_id": sid, "model": run["label"], "miou": result["miou"],
               "valid_pixels": result["valid_pixels"], "ignored_pixels": result["ignored_pixels"]}
        for class_id, iou in enumerate(result["iou_per_class"]):
            row[f"iou_class_{class_id}"] = iou
        metric_rows.append(row)
        if CROSS_CHECK_REPORTS and sid in report_rows:
            saved = report_rows[sid]
            for actual, expected in zip(
                    [result["miou"], *result["iou_per_class"]],
                    [saved["miou"], *saved["iou_per_class"]], strict=True):
                if ((actual is None) != (expected is None) or
                        (actual is not None and not np.isclose(actual, expected, rtol=0, atol=1e-12))):
                    raise ValueError(f"Saved per-sample metrics differ: {run['label']}/{sid}")
        elif CROSS_CHECK_REPORTS:
            warnings.warn(f"No per-sample report entry to cross-check: {run['label']}/{sid}")

metrics_table = pd.DataFrame(metric_rows)
display(pd.DataFrame([{"class_id": int(key), "class_name": value}
                      for key, value in mapping.items()]))
display(metrics_table.style.format(precision=4, na_rep="N/A"))


def metric_annotation(sample_id, model_index):
    result = metrics_by_sample[(sample_id, model_index)]
    def percent(value):
        return "N/A" if value is None else f"{100 * value:.2f}%"
    lines = []
    for name in DISPLAY_METRICS:
        if name == "miou":
            lines.append(f"mIoU: {percent(result['miou'])}")
        elif water_id is not None:
            lines.append(f"Water IoU: {percent(result['iou_per_class'][water_id])}")
    return "\n".join(lines)


## Plotting
Shared scientific_style covers construction and is also active through export and display.
Segmentation and error colours follow GeoTest's existing prediction figures. Ignored
target pixels remain grey; masks are rendered with nearest-neighbour interpolation.
Every image has the same panel box, preserving pixel aspect ratio. Metrics below each
model panel are per-chip results from the original numerical masks, including on error figures.
Binary errors use class 1 as positive; multiclass errors distinguish correct/incorrect pixels.


In [ ]:
@scientific_style()
def comparison_figure(error_maps=False):
    meta = runs[0]["metadata"]
    nclasses, ignore = meta["num_classes"], meta["ignore_index"]
    names = [meta["class_mapping"][str(i)] for i in range(nclasses)]
    names = [str(name).replace("_", " ").replace("non ", "non-").capitalize()
             if name is not None else f"Class {i}" for i, name in enumerate(names)]
    colours = (["#263440", "#20b9d2"] if nclasses == 2 else
               [plt.colormaps["tab20"](i % 20) for i in range(nclasses)])
    if nclasses > 20:
        warnings.warn("Class colours repeat beyond 20 classes.")
    cmap = ListedColormap(colours)
    cmap.set_bad("#c5c7cb")
    norm = BoundaryNorm(np.arange(nclasses + 1) - .5, nclasses)
    error_colours = (["#e5e9ed", "#f2a444", "#e55764", "#c5c7cb"] if nclasses == 2
                     else ["#e5e9ed", "#e55764", "#c5c7cb"])
    error_names = (["Correct", "False positive (class 1)", "False negative (class 1)", "Ignored"]
                   if nclasses == 2 else ["Correct", "Incorrect", "Ignored"])
    rgb_column = bool(rgb_images)
    titles = (["RGB context"] if rgb_column else []) + ["Ground truth"] + [
        run["label"] + (" error" if error_maps else "") for run in runs]
    fig, axes = plt.subplots(len(selected_ids), len(titles), squeeze=False,
                             figsize=(PANEL_INCHES * len(titles),
                                      (PANEL_INCHES + .35 * bool(DISPLAY_METRICS)) * len(selected_ids)))
    for row, sid in enumerate(selected_ids):
        target = samples[sid][0][0]
        valid = target != ignore
        offset = int(rgb_column)
        if rgb_column:
            axes[row, 0].imshow(rgb_images[sid], interpolation="nearest")
        axes[row, offset].imshow(np.ma.array(target, mask=~valid), cmap=cmap,
                                  norm=norm, interpolation="nearest")
        for column, (_, prediction) in enumerate(samples[sid], start=offset + 1):
            if error_maps:
                errors = np.zeros(target.shape, dtype=np.uint8)
                if nclasses == 2:
                    errors[valid & (prediction == 1) & (target == 0)] = 1
                    errors[valid & (prediction == 0) & (target == 1)] = 2
                else:
                    errors[valid & (prediction != target)] = 1
                errors[~valid] = len(error_colours) - 1
                axes[row, column].imshow(errors, cmap=ListedColormap(error_colours),
                                          vmin=0, vmax=len(error_colours) - 1,
                                          interpolation="nearest")
            else:
                axes[row, column].imshow(np.ma.array(prediction, mask=~valid),
                                          cmap=cmap, norm=norm, interpolation="nearest")
            axes[row, column].set_xlabel(
                metric_annotation(sid, column - offset - 1),
                fontsize=plt.rcParams["xtick.labelsize"], labelpad=4)
        for ax in axes[row]:
            ax.set_box_aspect(1)
            ax.set_anchor("N")
            ax.set_xticks([])
            ax.set_yticks([])
            ax.grid(False, which="both")
            for spine in ax.spines.values():
                spine.set_visible(False)
        axes[row, 0].set_ylabel(sid, fontsize=plt.rcParams["ytick.labelsize"])
    for ax, title in zip(axes[0], titles):
        ax.set_title(title)
    if SHOW_LEGEND:
        handles = [Patch(facecolor=c, label=name) for c, name in zip(colours, names)]
        if error_maps:
            handles += [Patch(facecolor=c, label=name)
                        for c, name in zip(error_colours, error_names)]
        else:
            handles.append(Patch(facecolor="#c5c7cb", label="Ignored"))
        fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(.5, 0),
                   ncol=max(1, min(LEGEND_COLUMNS, len(handles))), frameon=False)
    compact_legend_layout(fig, w_pad=.6)
    return fig


## Generate figures
PNG and PDF exports share the same figure. Re-running this cell replaces only these qualitative figures and their selection record, never evaluation outputs. Change `OUTPUT_DIR` to retain another selection.


In [ ]:
if SAVE_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with scientific_style():
    for errors in ([False, True] if MAKE_ERROR_FIGURE else [False]):
        figure = comparison_figure(error_maps=errors)
        if SAVE_FIGURES:
            save_figure(figure, OUTPUT_DIR / ("errors.png" if errors else "predictions.png"),
                        bbox_inches="tight")
        plt.show()
        plt.close(figure)

if SAVE_FIGURES:
    selection = {
        "dataset": runs[0]["dataset"], "split": SPLIT, "sample_ids": selected_ids,
        "selection": "manual" if SAMPLE_IDS else "random_common",
        "selection_seed": None if SAMPLE_IDS else SELECTION_SEED,
        "rgb_displayed": bool(rgb_images), "rgb_bands": list(RGB_BANDS),
        "rgb_sources": rgb_sources,
        "display_metrics": list(DISPLAY_METRICS), "water_class_id": water_id,
        "runs": [{"label": run["label"], "directory": str(run["directory"]),
                  "metadata": run["metadata"]} for run in runs],
    }
    (OUTPUT_DIR / "selection.json").write_text(
        json.dumps(selection, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved figures and selection record: {OUTPUT_DIR}")

if SAVE_FIGURES:
    metrics_table.to_csv(OUTPUT_DIR / "per_chip_metrics.csv", index=False, na_rep="N/A")
